# 3D face recognition from synthetic point clouds: full pipeline

This notebook runs the whole training pipeline from the paper *Learning Directly from Synthetic Point Clouds for "In-the-wild" 3D Face Recognition*:

| # | Stage | What it does | Output | Skipped when |
|---|---|---|---|---|
| 1 | **Generation** | Random BFM2017 faces → `[4096, 7]` point clouds (xyz, normals, curvature) | `Data/synthetic`, `Data/synthetic_test` | the folder has a `_COMPLETE` marker |
| 2 | **Classification** | Pre-train on 1000 synthetic identities with an A-Softmax head | `runs/pipeline/cls/` | `final.pth.tar` exists |
| 3 | **Triplet** | Fine-tune with randomly drawn (anchor, positive, negative) triplets | `runs/pipeline/triplet/` | `final.pth.tar` exists |
| 4 | **PK** | Fine-tune with P×K batches and online batch-hard mining | `runs/pipeline/pk/` | `final.pth.tar` exists |
| 5 | **Evaluation** | Compares every model on the same test set: table, ROC, CMC, score distributions, t-SNE, failure cases | `runs/pipeline/results.csv` | always runs |

**Checkpoints and skipping.** Every training stage writes three files:
* `last.pth.tar` after every epoch. If the session dies, re-running the stage resumes from it.
* `best.pth.tar`, the epoch with the best test score (`SELECT_METRIC`). Later stages start from this file.
* `final.pth.tar` when the stage has finished all its epochs. Its presence makes the stage skip.

To retrain a stage, delete its folder in `runs/pipeline/`.

**Test protocol.** All metrics are measured on identities that are **never seen in training**. That is either 200 extra synthetic identities, or a held-out 20% of the identities in `REAL_DATA`. For each identity, its first scan is the *gallery* entry and its other scans are *probes*.
* **rank-1**: the fraction of probes whose most similar gallery scan has the right identity.
* **CMC**: the same, counted within the top-k matches.
* **TAR@FAR**: the fraction of genuine pairs accepted at a threshold that accepts the given fraction of impostor pairs.
* **EER**: the error rate where false accepts equal false rejects.
* **AUC**: area under the ROC curve.

All of these use cosine similarity between L2-normalised 512-d embeddings.

Only reusable utilities live in the repository: the network, the data loaders, the face generator, the losses and the metrics. All stage logic and every training loop are in this notebook.

## 0. Setup
Clones the repository if needed, installs the dependencies and builds the CUDA ops. The build runs once and is skipped when the compiled `.so` file already exists.

In [ ]:
import os, sys, glob, subprocess

REPO_URL    = "https://github.com/AdarWa/3DFacePointCloudNet.git"
REPO_BRANCH = "claude/busy-ritchie-gyjpka"
REPO_DIR    = "/kaggle/working/3DFacePointCloudNet"

# If the notebook is opened from inside a checkout, use that checkout.
for candidate in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(candidate, "pointnet2")):
        REPO_DIR = candidate
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Repository:", REPO_DIR)

In [ ]:
!pip install -q -r requirements.txt plotly pandas
if glob.glob("pointnet2/_ext*.so"):
    print("CUDA ops already built")
else:
    !python setup.py build_ext --inplace

In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.manifold import TSNE

# Repository utilities
from pointnet2.models import Pointnet2FaceClsSSG
from pointnet2.train.layer import AngleLinear, MarginCosineProduct
from pointnet2.data import data_utils
from pointnet2.data.generate_faces import generate
from pointnet2.data.TripletNpyFaceDataset import (
    find_npy_files, group_by_subject, split_by_subject, split_subjects, groups_to_samples,
    SingleScanNpyDataset, TripletNpyFaceDataset, PKSampler)
from pointnet2.utils.face_eval import (
    triplet_loss, batch_hard_triplet_loss, extract_embeddings,
    gallery_probe_split, verification_metrics)

assert torch.cuda.is_available(), "This notebook needs a CUDA GPU (Kaggle: Settings > Accelerator > GPU)"
DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 1. Configuration
Every setting is in this one cell.

In [ ]:
# ---- paths ------------------------------------------------------------------
BFM_PATH      = "Data/model2017-1_face12_nomouth.h5"  # BFM2017 model, e.g. /kaggle/input/<dataset>/model2017-1_face12_nomouth.h5
SYN_TRAIN_DIR = "Data/synthetic"         # synthetic training identities
SYN_TEST_DIR  = "Data/synthetic_test"    # extra, unseen synthetic identities (test set when REAL_DATA is None)
REAL_DATA     = None                     # folder of real {subject}_{scan}.npy scans, or None
WORK_DIR      = "runs/pipeline"          # checkpoints, histories and results
PAPER_CKPT    = "checkpoints/20191028_1000cls_model_best.pth.tar"  # the authors' model (evaluated as a baseline)

# ---- data -------------------------------------------------------------------
NUM_POINTS      = 4096   # points per cloud fed to the network
CURVATURE_POWER = 0.1    # curvature channel is fed as curv ** 0.1, as in the paper
NUM_WORKERS     = 4
SEED            = 0

# ---- stage 1: generation ----------------------------------------------------
GEN = dict(num_subjects=1000, scans_per_subject=20, num_points=NUM_POINTS,
           noise_std=0.0,          # vertex noise in mm, to imitate a scanner
           neutral_first=True,     # scan 000 of every identity has a neutral expression
           seed=SEED)
GEN_TEST = dict(GEN, num_subjects=200, start_subject=GEN["num_subjects"])  # different identities
REAL_TEST_RATIO = 0.2    # REAL_DATA only: fraction of identities held out as the test set

# ---- stage 2: classification pre-training ----------------------------------
CLS = dict(epochs=10, batch_size=32, lr=1e-3, weight_decay=1e-5, lr_milestones=[5, 8, 9],
           head="AL",              # "AL" = A-Softmax (paper) | "MCP" = CosFace | "linear"
           val_ratio=0.1,          # scans of each training identity held out to measure accuracy
           augment=True)

# ---- stage 3: triplet fine-tuning (random triplets) ------------------------
TRIPLET = dict(init="cls",         # starting checkpoint: "cls" | "paper"
               epochs=10, triplets_per_epoch=20000, batch_size=16,   # 16 triplets = 48 clouds per step
               lr=5e-5, weight_decay=1e-5, lr_milestones=[6, 9], margin=0.3, augment=True)

# ---- stage 4: PK fine-tuning (batch-hard mining) ---------------------------
PK = dict(init="cls",              # starting checkpoint: "cls" | "triplet" | "paper"
          epochs=20, P=8, K=4, batches_per_epoch=500,                 # 8 identities x 4 scans = 32 clouds per step
          lr=5e-5, weight_decay=1e-5, lr_milestones=[12, 18], margin=0.3, augment=True)

# ---- evaluation -------------------------------------------------------------
SELECT_METRIC = "rank1"  # picks best.pth.tar (higher is better: rank1, auc, "tar@far=0.001", "tar@far=0.01")
EVAL_BATCH    = 64

## 2. Helpers
Shared building blocks for all stages: checkpoint files, model loading, augmentation, evaluation and plotting.

In [ ]:
# ---- checkpoints and skipping ----------------------------------------------
def stage_files(stage):
    """Files written by a training stage."""
    folder = os.path.join(WORK_DIR, stage)
    os.makedirs(folder, exist_ok=True)
    return {"last": os.path.join(folder, "last.pth.tar"),        # resume point (every epoch)
            "best": os.path.join(folder, "best.pth.tar"),        # best test score; used by later stages
            "final": os.path.join(folder, "final.pth.tar"),      # written when the stage finishes
            "history": os.path.join(folder, "history.json")}     # per-epoch metrics

def stage_done(stage):
    return os.path.isfile(stage_files(stage)["final"])

def save_backbone(path, model, **info):
    """Backbone checkpoint in the repository format (load_checkpoint / train_triplet.py can read it)."""
    torch.save(dict(model_state=model.state_dict(), **info), path)

def build_model():
    return Pointnet2FaceClsSSG(input_channels=3, use_xyz=True).to(DEVICE)

def load_backbone(path):
    model = build_model()
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt.get("model_state", ckpt))
    return model

def init_checkpoint(init):
    """Checkpoint a fine-tuning stage starts from: 'paper' or another stage's best.pth.tar."""
    path = PAPER_CKPT if init == "paper" else stage_files(init)["best"]
    assert os.path.isfile(path), f"Starting checkpoint for init='{init}' not found: {path}"
    return path

def resume_last(files, **parts):
    """Restore model/optimizer/scheduler from last.pth.tar. Returns (start_epoch, history, best)."""
    if not os.path.isfile(files["last"]):
        return 0, [], -1.0
    state = torch.load(files["last"], map_location="cpu", weights_only=False)
    for name, part in parts.items():
        part.load_state_dict(state[name])
    print(f"Resuming after epoch {state['epoch']}")
    return state["epoch"], state["history"], state["best"]

def finish_epoch(stage, files, row, history, best, **parts):
    """Log the epoch, update best.pth.tar and save the resume point. Returns the new best score."""
    history.append(row)
    print(" | ".join(f"{k} {v:.4g}" if isinstance(v, float) else f"{k} {v}" for k, v in row.items()))
    if row[SELECT_METRIC] > best:
        best = row[SELECT_METRIC]
        save_backbone(files["best"], parts["model"], stage=stage, metrics=row)
        print(f"  -> new best {SELECT_METRIC} = {best:.4f}, saved {files['best']}")
    torch.save(dict(epoch=row["epoch"], history=history, best=best,
                    **{name: part.state_dict() for name, part in parts.items()}), files["last"])
    with open(files["history"], "w") as f:
        json.dump(history, f, indent=1)
    return best

def load_history(stage):
    with open(stage_files(stage)["history"]) as f:
        return json.load(f)

def seed_worker(worker_id):
    np.random.seed(torch.initial_seed() % 2 ** 32)   # different random augmentation in every worker

class Mean:
    """Running average for the progress bars."""
    def __init__(self):
        self.total, self.count = 0.0, 0
    def add(self, value, n=1):
        self.total += value * n
        self.count += n
    @property
    def value(self):
        return self.total / max(self.count, 1)

In [ ]:
# ---- augmentation (training only) ------------------------------------------
# A random rotation of up to 10 degrees about each axis (applied to points and normals), then 0.002 xyz jitter.
_AUGMENTATIONS = [data_utils.PointcloudRotate(axis=np.array([1.0, 0.0, 0.0])),
                  data_utils.PointcloudRotate(axis=np.array([0.0, 1.0, 0.0])),
                  data_utils.PointcloudRotate(axis=np.array([0.0, 0.0, 1.0])),
                  data_utils.PointcloudJitter(std=0.002)]

def augment(points):
    for transform in _AUGMENTATIONS:
        points = transform(points)
    return points

# ---- evaluation -------------------------------------------------------------
def evaluate(model, test_set):
    """Embed the test gallery and probes; returns (metrics dict, curves dict)."""
    g_files, g_labels, p_files, p_labels = test_set
    kw = dict(num_points=NUM_POINTS, curvature_power=CURVATURE_POWER,
              batch_size=EVAL_BATCH, num_workers=NUM_WORKERS, device=DEVICE)
    g_feat = extract_embeddings(model, g_files, **kw)
    p_feat = extract_embeddings(model, p_files, **kw)
    return verification_metrics(g_feat, g_labels, p_feat, p_labels)

# ---- plots ------------------------------------------------------------------
def plot_history(history, title):
    """Per-epoch curves: training loss, batch statistics and test metrics."""
    panels = [("Training loss", ["train_loss"]),
              ("Batch statistics", ["train_acc", "val_acc", "active", "d_ap", "d_an"]),
              ("Test set (unseen identities)", ["rank1", "tar@far=0.001", "tar@far=0.01", "auc"])]
    epochs = [r["epoch"] for r in history]
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, (name, keys) in zip(axes, panels):
        for key in keys:
            if any(key in r for r in history):
                ax.plot(epochs, [r.get(key, np.nan) for r in history], marker="o", label=key)
        ax.set_title(name)
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

FRONT_VIEW = dict(eye=dict(x=0, y=0, z=2.2), up=dict(x=0, y=1, z=0))

def show_clouds(clouds, titles, color="curvature", height=420):
    """Interactive 3D view of [N, 7] clouds side by side. color: 'curvature' | 'normals' (or a list, one per cloud)."""
    colors = color if isinstance(color, list) else [color] * len(clouds)
    fig = make_subplots(rows=1, cols=len(clouds), specs=[[{"type": "scene"}] * len(clouds)],
                        subplot_titles=titles)
    for i, (cloud, mode) in enumerate(zip(clouds, colors), start=1):
        if mode == "normals":   # normal direction as RGB: (n + 1) / 2
            rgb = ((cloud[:, 3:6] + 1) / 2 * 255).clip(0, 255).astype(int)
            marker = dict(size=2, color=[f"rgb({r},{g},{b})" for r, g, b in rgb])
        else:                   # curvature, as the network sees it
            marker = dict(size=2, color=np.clip(cloud[:, 6], 0, None) ** CURVATURE_POWER,
                          colorscale="Viridis")
        fig.add_trace(go.Scatter3d(x=cloud[:, 0], y=cloud[:, 1], z=cloud[:, 2],
                                   mode="markers", marker=marker), row=1, col=i)
        fig.update_layout(**{"scene" + ("" if i == 1 else str(i)): dict(
            aspectmode="data", camera=FRONT_VIEW,
            xaxis_visible=False, yaxis_visible=False, zaxis_visible=False)})
    fig.update_layout(height=height, showlegend=False, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

## 3. Stage 1: generate synthetic faces
Every identity is one random **shape** drawn from the BFM2017 morphable model. Every scan of that identity adds a random **expression** (scan 000 is neutral). For each face mesh the generator then:
1. computes vertex normals from the triangles, pointing towards the camera (+z);
2. computes curvature as PCL does: the surface variation λ0 / (λ0 + λ1 + λ2) over each vertex's 20 nearest neighbours;
3. samples `NUM_POINTS` vertices, centres them and scales them into the unit sphere.

The output is `<dir>/<subject>/<subject>_<scan>.npy`. Generation runs in parallel and is deterministic per identity. It also resumes: identities that are already complete are not generated again.

In [ ]:
def generate_once(out_dir, options):
    """Generate a synthetic set unless its _COMPLETE marker exists."""
    marker = os.path.join(out_dir, "_COMPLETE")
    if os.path.isfile(marker):
        print(f"[skip] {out_dir} is already generated")
        return
    assert os.path.isfile(BFM_PATH), (f"BFM2017 model not found at {BFM_PATH}. Download "
                                      "model2017-1_face12_nomouth.h5 from https://faces.dmi.unibas.ch/bfm/bfm2017.html")
    generate(BFM_PATH, out_dir, **options)
    with open(marker, "w") as f:
        json.dump(options, f, indent=1)

generate_once(SYN_TRAIN_DIR, GEN)
if REAL_DATA is None:
    generate_once(SYN_TEST_DIR, GEN_TEST)

### 3.1 Inspect the generated data
The views are interactive: drag to rotate, scroll to zoom. On correct data, curvature is high on the nose, eye corners, lips and chin and low on the cheeks and forehead. In the normals view, colour should change smoothly across the face.

In [ ]:
train_files = find_npy_files(SYN_TRAIN_DIR)
preview_groups = group_by_subject(train_files)
example = np.load(train_files[0])

print(f"{len(preview_groups)} identities, {len(train_files)} clouds of shape {example.shape} "
      f"(~{len(train_files) * os.path.getsize(train_files[0]) / 1e9:.2f} GB)")
print("channels         : 0-2 xyz | 3-5 normal | 6 curvature")
print("max xyz radius   :", np.linalg.norm(example[:, :3], axis=1).max().round(3), "(unit sphere -> 1.0)")
print("mean |normal|    :", np.linalg.norm(example[:, 3:6], axis=1).mean().round(3), "(unit vectors -> 1.0)")
print("curvature range  :", example[:, 6].min().round(4), "to", example[:, 6].max().round(4), "(0 = flat, max 1/3)")

show_clouds([example, example], ["curvature (curv^0.1)", "normals as RGB"], color=["curvature", "normals"])

In [ ]:
# Four different identities (neutral scans), then four expressions of the same identity
subjects = list(preview_groups)[:4]
show_clouds([np.load(preview_groups[s][0]) for s in subjects], subjects)

same_person = preview_groups[subjects[0]][:4]
show_clouds([np.load(f) for f in same_person], [os.path.basename(f) for f in same_person])

In [ ]:
# Curvature distribution over 50 random clouds: raw values and the curv^0.1 values the network sees
rng = np.random.RandomState(SEED)
curv = np.concatenate([np.load(f)[:, 6] for f in rng.choice(train_files, min(50, len(train_files)), replace=False)])
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].hist(curv, bins=100, color="tab:blue"); axes[0].set_title("raw curvature")
axes[1].hist(np.clip(curv, 0, None) ** CURVATURE_POWER, bins=100, color="tab:green")
axes[1].set_title(f"curvature ** {CURVATURE_POWER} (network input)")
plt.tight_layout(); plt.show()

## 4. Datasets and splits
* **Classification** trains on the synthetic training identities. From each identity, `CLS["val_ratio"]` of its scans are held out to measure classification accuracy.
* **Triplet and PK** fine-tune on `REAL_DATA` when it is set, without its held-out identities. Otherwise they fine-tune on the synthetic training identities.
* **Test set** has no identities in common with any training set. It is the synthetic test identities, or the held-out real identities when `REAL_DATA` is set.

In [ ]:
syn_groups = group_by_subject(find_npy_files(SYN_TRAIN_DIR), min_samples=2)
cls_train_groups, cls_val_groups = split_by_subject(syn_groups, CLS["val_ratio"], SEED)

if REAL_DATA is None:
    finetune_groups = syn_groups
    test_groups = group_by_subject(find_npy_files(SYN_TEST_DIR), min_samples=2)
else:
    real_groups = group_by_subject(find_npy_files(REAL_DATA), min_samples=2)
    finetune_groups, test_groups = split_subjects(real_groups, REAL_TEST_RATIO, SEED)

assert not set(test_groups) & (set(syn_groups) | set(finetune_groups)), "test identities leak into training"
TEST_SET = gallery_probe_split(test_groups)

def describe(groups):
    return dict(identities=len(groups), scans=sum(len(v) for v in groups.values()))

display(pd.DataFrame({
    "classification train": describe(cls_train_groups),
    "classification held-out scans": describe(cls_val_groups),
    "fine-tuning train (triplet / PK)": describe(finetune_groups),
    "test (unseen identities)": describe(test_groups),
}).T.assign(**{"gallery / probes": ["", "", "", f"{len(TEST_SET[0])} / {len(TEST_SET[2])}"]}))

## 5. Stage 2: classification pre-training
Each synthetic identity is one class. The network maps a cloud to a 512-d embedding, and a classification head maps the embedding to class scores. The default head is **A-Softmax** (`AngleLinear`), as in the paper. It adds an angular margin between classes, so embeddings of one identity cluster tightly on the unit sphere. Only the backbone is kept for the later stages.

Logged every epoch:
* `train_loss`, and `train_acc` measured without the margin;
* `val_acc`, the accuracy on held-out scans of the training identities;
* every test metric, measured on unseen identities. These show whether the embedding generalises.

In [ ]:
def make_head(kind, num_classes):
    heads = {"AL": lambda: AngleLinear(512, num_classes),
             "MCP": lambda: MarginCosineProduct(512, num_classes),
             "linear": lambda: nn.Linear(512, num_classes, bias=False)}
    return heads[kind]().to(DEVICE)

def margin_logits(head, feat, labels):
    """Logits for the loss, with the angular margin applied for AL / MCP."""
    return head(feat) if isinstance(head, nn.Linear) else head(feat, labels)

def plain_logits(head, feat):
    """Margin-free logits for accuracy (cosine to each class centre)."""
    if isinstance(head, nn.Linear):
        return head(feat)
    return F.linear(F.normalize(feat), F.normalize(head.weight))

@torch.no_grad()
def classification_accuracy(model, head, loader):
    model.eval()
    correct = total = 0
    for points, labels in loader:
        points, labels = points.to(DEVICE), labels.to(DEVICE)
        correct += (plain_logits(head, model(points)).argmax(1) == labels).sum().item()
        total += len(labels)
    return correct / max(total, 1)

In [ ]:
def train_classification():
    files = stage_files("cls")

    # ---- data: one class per identity ----
    train_samples, subjects = groups_to_samples(cls_train_groups)            # [(file, class index)]
    class_of = {s: i for i, s in enumerate(subjects)}
    val_samples = [(f, class_of[s]) for s, fs in cls_val_groups.items() for f in fs]
    train_loader = DataLoader(
        SingleScanNpyDataset(train_samples, NUM_POINTS, CURVATURE_POWER,
                             transforms=augment if CLS["augment"] else None),
        batch_size=CLS["batch_size"], shuffle=True, drop_last=True,
        num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker)
    val_loader = DataLoader(SingleScanNpyDataset(val_samples, NUM_POINTS, CURVATURE_POWER),
                            batch_size=EVAL_BATCH, num_workers=NUM_WORKERS, pin_memory=True)

    # ---- model, head, optimiser ----
    model = build_model()
    head = make_head(CLS["head"], len(subjects))
    optimizer = torch.optim.Adam([*model.parameters(), *head.parameters()],
                                 lr=CLS["lr"], weight_decay=CLS["weight_decay"])
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, CLS["lr_milestones"], gamma=0.1)
    start_epoch, history, best = resume_last(files, model=model, head=head,
                                             optimizer=optimizer, scheduler=scheduler)
    if hasattr(head, "iter"):   # A-Softmax anneals its margin per training step
        head.iter = start_epoch * len(train_loader)
    print(f"{len(subjects)} classes | {len(train_samples)} train scans | "
          f"{len(val_samples)} held-out scans | {len(train_loader)} steps per epoch")

    # ---- training loop ----
    for epoch in range(start_epoch, CLS["epochs"]):
        model.train(); head.train()
        loss_avg, acc_avg = Mean(), Mean()
        progress = tqdm(train_loader, desc=f"classification {epoch + 1}/{CLS['epochs']}")
        for points, labels in progress:
            points = points.to(DEVICE, non_blocking=True)        # [B, N, 7]
            labels = labels.to(DEVICE, non_blocking=True)        # [B]

            embeddings = model(points)                           # [B, 512]
            loss = F.cross_entropy(margin_logits(head, embeddings, labels), labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            accuracy = (plain_logits(head, embeddings).argmax(1) == labels).float().mean().item()
            loss_avg.add(loss.item(), len(labels))
            acc_avg.add(accuracy, len(labels))
            progress.set_postfix(loss=f"{loss_avg.value:.3f}", acc=f"{acc_avg.value:.3f}")
        lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        # ---- end of epoch: metrics, checkpoints ----
        test_metrics, _ = evaluate(model, TEST_SET)
        row = dict(epoch=epoch + 1, lr=lr, train_loss=loss_avg.value, train_acc=acc_avg.value,
                   val_acc=classification_accuracy(model, head, val_loader), **test_metrics)
        best = finish_epoch("cls", files, row, history, best, model=model, head=head,
                            optimizer=optimizer, scheduler=scheduler)

    save_backbone(files["final"], model, stage="cls", epochs=CLS["epochs"])
    torch.save(head.state_dict(), os.path.join(os.path.dirname(files["final"]), "head.pth"))

In [ ]:
if stage_done("cls"):
    print(f"[skip] classification already finished ({stage_files('cls')['final']})")
else:
    train_classification()
plot_history(load_history("cls"), "Stage 2: classification pre-training")

## 6. Stage 3: triplet fine-tuning (random triplets)
A fresh set of `triplets_per_epoch` random triplets is drawn every epoch. Each triplet is an anchor, a positive (another scan of the same identity) and a negative (a scan of a different identity). The loss pulls the anchor towards the positive and pushes it away from the negative, until the negative is at least `margin` further away:

  `loss = max(0, d(a, p) - d(a, n) + margin)`, using Euclidean distance between L2-normalised embeddings.

The three clouds of every triplet go through the network in a single forward pass. Logged: `active` (the fraction of triplets that still violate the margin), plus the mean `d_ap` and `d_an`. Epoch 0 is the starting checkpoint, scored before any fine-tuning.

In [ ]:
def train_triplet():
    files = stage_files("triplet")
    dataset = TripletNpyFaceDataset(finetune_groups, n_triplets=TRIPLET["triplets_per_epoch"],
                                    n_points=NUM_POINTS, curvature_power=CURVATURE_POWER,
                                    transforms=augment if TRIPLET["augment"] else None)
    loader = DataLoader(dataset, batch_size=TRIPLET["batch_size"], shuffle=True, drop_last=True,
                        num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker)

    model = load_backbone(init_checkpoint(TRIPLET["init"]))
    optimizer = torch.optim.Adam(model.parameters(), lr=TRIPLET["lr"], weight_decay=TRIPLET["weight_decay"])
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, TRIPLET["lr_milestones"], gamma=0.1)
    start_epoch, history, best = resume_last(files, model=model, optimizer=optimizer, scheduler=scheduler)

    if not history:   # epoch 0: score the starting checkpoint
        best = finish_epoch("triplet", files, dict(epoch=0, **evaluate(model, TEST_SET)[0]), history, best,
                            model=model, optimizer=optimizer, scheduler=scheduler)

    for epoch in range(start_epoch, TRIPLET["epochs"]):
        if epoch > 0:
            dataset.regenerate()                                 # new random triplets every epoch
        model.train()
        loss_avg, active_avg, dap_avg, dan_avg = Mean(), Mean(), Mean(), Mean()
        progress = tqdm(loader, desc=f"triplet {epoch + 1}/{TRIPLET['epochs']}")
        for anchor, positive, negative, _, _ in progress:
            clouds = torch.cat([anchor, positive, negative]).to(DEVICE, non_blocking=True)  # [3B, N, 7]
            emb_anchor, emb_positive, emb_negative = model(clouds).chunk(3)                 # one forward pass
            loss, stats = triplet_loss(emb_anchor, emb_positive, emb_negative, TRIPLET["margin"])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            n = len(anchor)
            loss_avg.add(loss.item(), n); active_avg.add(stats["active"], n)
            dap_avg.add(stats["d_ap"], n); dan_avg.add(stats["d_an"], n)
            progress.set_postfix(loss=f"{loss_avg.value:.3f}", active=f"{active_avg.value:.2f}",
                                 d_ap=f"{dap_avg.value:.3f}", d_an=f"{dan_avg.value:.3f}")
        lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        row = dict(epoch=epoch + 1, lr=lr, train_loss=loss_avg.value, active=active_avg.value,
                   d_ap=dap_avg.value, d_an=dan_avg.value, **evaluate(model, TEST_SET)[0])
        best = finish_epoch("triplet", files, row, history, best,
                            model=model, optimizer=optimizer, scheduler=scheduler)

    save_backbone(files["final"], model, stage="triplet", epochs=TRIPLET["epochs"])

In [ ]:
if stage_done("triplet"):
    print(f"[skip] triplet training already finished ({stage_files('triplet')['final']})")
else:
    train_triplet()
plot_history(load_history("triplet"), "Stage 3: triplet fine-tuning")

## 7. Stage 4: PK fine-tuning (online batch-hard mining)
Each batch holds **P identities × K scans** of each identity. Identities with fewer than K scans are sampled with replacement. Every scan in the batch acts as an anchor and is trained against:
* its **hardest positive**: the furthest scan of the same identity in the batch;
* its **hardest negative**: the closest scan of another identity in the batch.

The loss is averaged over the anchors that still violate the margin, so easy anchors don't dilute the gradient. Batch-hard gradients are sharper than random-triplet ones, which is why the learning rate is low. Logged: `active`, the fraction of anchors that violate the margin. If it drops below about 0.1, consider a larger margin.

In [ ]:
def train_pk():
    files = stage_files("pk")
    train_samples, _ = groups_to_samples(finetune_groups)
    dataset = SingleScanNpyDataset(train_samples, NUM_POINTS, CURVATURE_POWER,
                                   transforms=augment if PK["augment"] else None)
    sampler = PKSampler(dataset.labels, p=PK["P"], k=PK["K"], batches_per_epoch=PK["batches_per_epoch"])
    loader = DataLoader(dataset, batch_sampler=sampler, num_workers=NUM_WORKERS,
                        pin_memory=True, worker_init_fn=seed_worker)

    model = load_backbone(init_checkpoint(PK["init"]))
    optimizer = torch.optim.Adam(model.parameters(), lr=PK["lr"], weight_decay=PK["weight_decay"])
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, PK["lr_milestones"], gamma=0.1)
    start_epoch, history, best = resume_last(files, model=model, optimizer=optimizer, scheduler=scheduler)

    if not history:   # epoch 0: score the starting checkpoint
        best = finish_epoch("pk", files, dict(epoch=0, **evaluate(model, TEST_SET)[0]), history, best,
                            model=model, optimizer=optimizer, scheduler=scheduler)

    print(f"P={sampler.p} identities x K={sampler.k} scans = {sampler.batch_size} clouds per step, "
          f"{len(sampler)} steps per epoch")
    for epoch in range(start_epoch, PK["epochs"]):
        model.train()
        loss_avg, active_avg = Mean(), Mean()
        progress = tqdm(loader, desc=f"PK {epoch + 1}/{PK['epochs']}")
        for points, labels in progress:
            points = points.to(DEVICE, non_blocking=True)        # [P*K, N, 7]
            labels = labels.to(DEVICE, non_blocking=True)        # [P*K]

            embeddings = model(points)                           # one forward pass for the whole batch
            loss, num_hard = batch_hard_triplet_loss(embeddings, labels, margin=PK["margin"])

            if num_hard > 0:                                     # nothing to learn from an all-easy batch
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            loss_avg.add(loss.item(), len(labels))
            active_avg.add(num_hard / len(labels), len(labels))
            progress.set_postfix(loss=f"{loss_avg.value:.3f}", active=f"{active_avg.value:.2f}")
        lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        row = dict(epoch=epoch + 1, lr=lr, train_loss=loss_avg.value, active=active_avg.value,
                   **evaluate(model, TEST_SET)[0])
        best = finish_epoch("pk", files, row, history, best,
                            model=model, optimizer=optimizer, scheduler=scheduler)

    save_backbone(files["final"], model, stage="pk", epochs=PK["epochs"])

In [ ]:
if stage_done("pk"):
    print(f"[skip] PK training already finished ({stage_files('pk')['final']})")
else:
    train_pk()
plot_history(load_history("pk"), "Stage 4: PK fine-tuning")

## 8. Stage 5: evaluation and comparison
Scores the `best.pth.tar` of every stage, plus the authors' original checkpoint, on the same test set.

The authors' checkpoint was trained on clouds centred on the nose tip and scaled to mm / 100, not on the unit-sphere clouds used here. It is listed only as a reference point.

In [ ]:
candidates = {"paper checkpoint": PAPER_CKPT,
              "classification": stage_files("cls")["best"],
              "triplet": stage_files("triplet")["best"],
              "PK": stage_files("pk")["best"]}
results, curves = {}, {}
for name, path in candidates.items():
    if not os.path.isfile(path):
        print(f"[skip] {name}: {path} not found")
        continue
    model = load_backbone(path)
    results[name], curves[name] = evaluate(model, TEST_SET)
    del model
    torch.cuda.empty_cache()

table = pd.DataFrame(results).T
table.to_csv(os.path.join(WORK_DIR, "results.csv"))
best_name = table[SELECT_METRIC].idxmax()
print(f"Test set: {len(TEST_SET[0])} identities, {len(TEST_SET[2])} probes. Best by {SELECT_METRIC}: {best_name}")
display(table.round(4))

In [ ]:
# ROC (verification) and CMC (identification) for every model
fig, (ax_roc, ax_cmc) = plt.subplots(1, 2, figsize=(15, 5))
for name, c in curves.items():
    ax_roc.semilogx(np.maximum(c["fpr"], 1e-6), c["tpr"], label=f"{name} (AUC {results[name]['auc']:.3f})")
    ax_cmc.plot(np.arange(1, len(c["cmc"]) + 1), c["cmc"], marker="o", label=name)
ax_roc.axvline(1e-3, color="gray", ls="--", lw=1)
ax_roc.set(title="ROC", xlabel="false accept rate (log)", ylabel="true accept rate", xlim=(1e-5, 1))
ax_cmc.set(title="CMC", xlabel="rank", ylabel="identification rate")
for ax in (ax_roc, ax_cmc):
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Genuine vs impostor similarity scores of the best model: the less they overlap, the better
c = curves[best_name]
plt.figure(figsize=(10, 4))
plt.hist(c["impostor"], bins=100, density=True, alpha=0.6, label="impostor pairs (different identity)")
plt.hist(c["genuine"], bins=100, density=True, alpha=0.6, label="genuine pairs (same identity)")
plt.xlabel("cosine similarity"); plt.ylabel("density"); plt.legend()
plt.title(f"Score distributions: {best_name}  (EER {results[best_name]['eer']:.3f})")
plt.show()

In [ ]:
# t-SNE of the best model's embeddings for 15 test identities (one colour per identity)
best_model = load_backbone(candidates[best_name])
shown = sorted(test_groups)[:15]
files = [f for s in shown for f in test_groups[s]]
identity = np.array([i for i, s in enumerate(shown) for _ in test_groups[s]])
embeddings = extract_embeddings(best_model, files, NUM_POINTS, CURVATURE_POWER, EVAL_BATCH, NUM_WORKERS, DEVICE)
xy = TSNE(n_components=2, perplexity=min(30, len(files) - 1), init="pca", random_state=SEED).fit_transform(embeddings)

plt.figure(figsize=(8, 7))
plt.scatter(xy[:, 0], xy[:, 1], c=identity, cmap="tab20", s=25)
plt.title(f"t-SNE of test embeddings: {best_name}"); plt.xticks([]); plt.yticks([])
plt.show()

In [ ]:
# Failure cases: probes whose closest gallery scan is a different identity
g_files, g_labels, p_files, p_labels = TEST_SET
kw = dict(num_points=NUM_POINTS, curvature_power=CURVATURE_POWER, batch_size=EVAL_BATCH,
          num_workers=NUM_WORKERS, device=DEVICE)
similarity = extract_embeddings(best_model, g_files, **kw) @ extract_embeddings(best_model, p_files, **kw).T
predicted = similarity.argmax(axis=0)
wrong = np.flatnonzero(np.array(g_labels)[predicted] != np.array(p_labels))
print(f"{len(wrong)} of {len(p_files)} probes misidentified by {best_name}")

for i in wrong[:3]:
    true_gallery = g_files[g_labels.index(p_labels[i])]
    show_clouds([np.load(p_files[i]), np.load(true_gallery), np.load(g_files[predicted[i]])],
                [f"probe {os.path.basename(p_files[i])}",
                 f"true identity (sim {similarity[g_labels.index(p_labels[i]), i]:.3f})",
                 f"predicted: {g_labels[predicted[i]]} (sim {similarity[predicted[i], i]:.3f})"],
                height=360)

## Outputs
* `runs/pipeline/<stage>/best.pth.tar`: the best backbone of each stage, in the repository checkpoint format. Load it with `load_backbone(path)`, or pass it to `python -m pointnet2.train.train_triplet -model_checkpoint ...`.
* `runs/pipeline/<stage>/history.json`: per-epoch metrics.
* `runs/pipeline/results.csv`: the comparison table above.

To run another experiment, change `WORK_DIR`, or delete one stage's folder and re-run the notebook. Every stage that has already finished is skipped.